In [102]:
import time
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

Web scraping : hatla2ee

In [103]:
url_1 = "https://eg.hatla2ee.com/en/car/search"  #[cite: 14]
headers = {"User-Agent": "Chrome/120.0"}  #

response_1 = requests.get(url_1, headers=headers, timeout=10)  #[cite: 5]
response_1.raise_for_status()  #[cite: 5]

soup_1 = BeautifulSoup(response_1.content, "html.parser")  #[cite: 5]
links_1 = soup_1.find_all("a", href=lambda h: h and "/car/" in h)  #[cite: 14]

car_ads_1 = [
    a
    for a in links_1
    if a.get("href")
    and ("/car/" in a.get("href"))
    and any(ch.isdigit() for ch in a.get("href").split("/")[-1])
]  #[cite: 16]

site1_rows = []
for a in car_ads_1:
  href = a.get("href", "")
  if "/price-limit/" in href:
    continue  #[cite: 16]

  parts = [p.strip() for p in a.get_text("|", strip=True).split("|") if p.strip()]  #[cite: 16]
  car_model = parts[0] if len(parts) > 0 else None  #[cite: 16]
  listings_raw = parts[1] if len(parts) > 1 else None  #[cite: 16]

  site1_rows.append({
      "Car_Model": car_model,
      "Listings_Count": listings_raw,
      "URL": "https://eg.hatla2ee.com" + href,
      "Source": "Hatla2ee",  #
  })

df_site1 = pd.DataFrame(site1_rows)  #[cite: 5]
print("Site 1 shape:", df_site1.shape)
df_site1.head()

Site 1 shape: (87, 4)


,Car_Model,Listings_Count,URL,Source
0,Mercedes C 180,(,https://eg.hatla2ee.com/en/car/mercedes/c-180,Hatla2ee
1,Mercedes E 200,(,https://eg.hatla2ee.com/en/car/mercedes/e-200,Hatla2ee
2,MG 5,(,https://eg.hatla2ee.com/en/car/moris-garage/mg-5,Hatla2ee
3,1600 CC,(,https://eg.hatla2ee.com/en/car/cc/1600,Hatla2ee
4,1500 CC,(,https://eg.hatla2ee.com/en/car/cc/1500,Hatla2ee


Web scraping : Contact car

In [104]:
url_2 = "https://www.contactcars.com/en/new-cars"
response_2 = requests.get(url_2, headers=headers, timeout=10)
response_2.raise_for_status()

soup_2 = BeautifulSoup(response_2.content, "html.parser")
all_links_2 = soup_2.find_all("a", href=True)

site2_rows = []
for a in all_links_2:
  href = a.get("href", "")
  if "/en/new-cars/" in href:
    parts = [
        p.strip() for p in a.get_text("|", strip=True).split("|") if p.strip()
    ]
    brand_name = parts[0] if len(parts) > 0 else None
    count_raw = parts[1] if len(parts) > 1 else None

    full_url = (
        href if href.startswith("http") else "https://www.contactcars.com" + href
    )

    site2_rows.append({
        "Car_Model": brand_name,
        "Listings_Count": count_raw,
        "URL": full_url,
        "Source": "ContactCars",
    })

time.sleep(1)

df_site2 = pd.DataFrame(site2_rows).drop_duplicates(subset=["URL"])
print("df_site2 shape:", df_site2.shape)
df_site2.head()

df_site2 shape: (36, 4)


,Car_Model,Listings_Count,URL,Source
0,Skoda,(,https://www.contactcars.com/en/new-cars/skoda,ContactCars
1,Mercedes,(,https://www.contactcars.com/en/new-cars/mercedes,ContactCars
2,MG,(,https://www.contactcars.com/en/new-cars/mg,ContactCars
3,Chery,(,https://www.contactcars.com/en/new-cars/chery,ContactCars
4,Nissan,(,https://www.contactcars.com/en/new-cars/nissan,ContactCars


Data Combination

In [105]:
df_combined = pd.concat([df_site1, df_site2], ignore_index=True)
print("df_combined shape:", df_combined.shape)
print(df_combined["Source"].value_counts())
df_combined.head()

df_combined shape: (123, 4)
Source
Hatla2ee       87
ContactCars    36
Name: count, dtype: int64


,Car_Model,Listings_Count,URL,Source
0,Mercedes C 180,(,https://eg.hatla2ee.com/en/car/mercedes/c-180,Hatla2ee
1,Mercedes E 200,(,https://eg.hatla2ee.com/en/car/mercedes/e-200,Hatla2ee
2,MG 5,(,https://eg.hatla2ee.com/en/car/moris-garage/mg-5,Hatla2ee
3,1600 CC,(,https://eg.hatla2ee.com/en/car/cc/1600,Hatla2ee
4,1500 CC,(,https://eg.hatla2ee.com/en/car/cc/1500,Hatla2ee


Data processing & cleaning 

In [106]:
import numpy as np
import pandas as pd



df_clean = df_combined[
    ~df_combined["URL"].astype(str).str.contains("/cc/", na=False)
].copy()


df_clean["Car_Model"] = (
    df_clean["Car_Model"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)
df_clean["Car_Model"] = df_clean["Car_Model"].replace(
    ["Nan", "None", ""], np.nan
)
df_clean["Car_Model"] = df_clean["Car_Model"].fillna("Unknown")


df_clean["Source"] = df_clean["Source"].astype(str).str.strip()

digits_only = (
    df_clean["Listings_Count"].astype(str).str.replace(r"[^0-9]", "", regex=True)
)
digits_only = digits_only.where(digits_only != "", np.nan)
df_clean["Listings_Count"] = pd.to_numeric(digits_only, errors="coerce")

if df_clean["Listings_Count"].notna().any():
  df_clean["Listings_Count"] = (
      df_clean["Listings_Count"]
      .fillna(df_clean["Listings_Count"].median())
      .astype(int)
  )
else:
  df_clean["Listings_Count"] = 0
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
df_clean.to_csv("cars_combined_cleaned.csv", index=False)

print("Clean shape:", df_clean.shape)
print(df_clean["Source"].value_counts())
df_clean.head(10)

Clean shape: (100, 4)
Source
Hatla2ee       64
ContactCars    36
Name: count, dtype: int64


,Car_Model,Listings_Count,URL,Source
0,Mercedes C 180,0,https://eg.hatla2ee.com/en/car/mercedes/c-180,Hatla2ee
1,Mercedes E 200,0,https://eg.hatla2ee.com/en/car/mercedes/e-200,Hatla2ee
2,Mg 5,0,https://eg.hatla2ee.com/en/car/moris-garage/mg-5,Hatla2ee
3,Cla 200,0,https://eg.hatla2ee.com/en/car/mercedes/cla-200,Hatla2ee
4,View All,0,https://eg.hatla2ee.com/en/car/mercedes/Merced...,Hatla2ee
5,Mercedes Gle 450 2022,0,https://eg.hatla2ee.com/en/car/mercedes/Merced...,Hatla2ee
6,Gle 450,0,https://eg.hatla2ee.com/en/car/mercedes/Merced...,Hatla2ee
7,View All,0,https://eg.hatla2ee.com/en/car/mercedes/c-180/...,Hatla2ee
8,Mercedes C 180 2026,0,https://eg.hatla2ee.com/en/car/mercedes/c-180/...,Hatla2ee
9,C 180,0,https://eg.hatla2ee.com/en/car/mercedes/c-180,Hatla2ee


BEFORE PREPROCESSING

In [107]:
print(" ## BEFORE PREPROCESSING ## ")
print(df_combined.head())
print("\n--- info ---")
df_combined.info()
print("\n--- describe ---")
print(df_combined.describe(include="all"))
print("\n--- nulls ---")
print(df_combined.isnull().sum())

 ## BEFORE PREPROCESSING ## 
        Car_Model Listings_Count  \
0  Mercedes C 180              (   
1  Mercedes E 200              (   
2            MG 5              (   
3         1600 CC              (   
4         1500 CC              (   

                                                URL    Source  
0     https://eg.hatla2ee.com/en/car/mercedes/c-180  Hatla2ee  
1     https://eg.hatla2ee.com/en/car/mercedes/e-200  Hatla2ee  
2  https://eg.hatla2ee.com/en/car/moris-garage/mg-5  Hatla2ee  
3            https://eg.hatla2ee.com/en/car/cc/1600  Hatla2ee  
4            https://eg.hatla2ee.com/en/car/cc/1500  Hatla2ee  

--- info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Car_Model       121 non-null    object
 1   Listings_Count  61 non-null     object
 2   URL             123 non-null    object
 3   Source          123 non-nul

AFTER PREPROCESSING

In [108]:
print("## AFTER PREPROCESSING ##")
print(df_clean.head())
print("\n--- info ---")
df_clean.info()
print("\n--- describe ---")
print(df_clean.describe(include="all"))
print("\n--- nulls ---")
print(df_clean.isnull().sum())

## AFTER PREPROCESSING ##
        Car_Model  Listings_Count  \
0  Mercedes C 180               0   
1  Mercedes E 200               0   
2            Mg 5               0   
3         Cla 200               0   
4        View All               0   

                                                 URL    Source  
0      https://eg.hatla2ee.com/en/car/mercedes/c-180  Hatla2ee  
1      https://eg.hatla2ee.com/en/car/mercedes/e-200  Hatla2ee  
2   https://eg.hatla2ee.com/en/car/moris-garage/mg-5  Hatla2ee  
3    https://eg.hatla2ee.com/en/car/mercedes/cla-200  Hatla2ee  
4  https://eg.hatla2ee.com/en/car/mercedes/Merced...  Hatla2ee  

--- info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Car_Model       100 non-null    object
 1   Listings_Count  100 non-null    int64 
 2   URL             100 non-null    object
 3   Source          100